# 3. Preprocessing: The Masterpiece Feature Suite (132 Dimensions)

This notebook builds the complete, production-grade feature engineering pipeline for the **Transferable UnitModel**.

### Feature Suite Architecture:
1. **Intra-Annual & Multi-Observation Variance (360° View):** Captures seasonal/angular variability across `obs_1`, `obs_2`, `obs_3`.
2. **Atmospheric De-Noising:** Filters undetected haze and cirrus anomalies (`Blue > 15,000`).
3. **Gap-Filling:** Interpolates missing annual observations to form an unbroken 1984–2024 trajectory.
4. **Engineered Spectral Indices:** Computes NDVI, NDBI, UI, MNDWI, and BSI.
5. **Spatial Urban Morphology:** Extracts local neighborhood spatial context using KD-Tree on `(px_key, py_key)`.
6. **Multi-Era Trajectory Aggregation:** Early (1984–2003), Late (2004–2024), and Year-on-Year change rates.
7. **Leakage-Free Standardisation & Export:** Standardised on Madrid and exported to `preprocessed_data.pkl`.

In [ ]:
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KDTree

# ── Constants ─────────────────────────────────────────────────────────────────
BASE_BANDS   = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
VAR_BANDS    = [f'{b}_var' for b in BASE_BANDS]
SPATIAL_BANDS = [f'{b}_spatial' for b in BASE_BANDS]
ALL_BANDS    = BASE_BANDS + VAR_BANDS + SPATIAL_BANDS
INDEX_COLS   = ['NDVI', 'NDBI', 'UI', 'MNDWI', 'BSI']
BLUE_MAX     = 15_000   # Atmospheric outlier threshold

CLASS_EVENTS = {'Amsterdam': 1945, 'Madrid': 1960}
CLASS_COLORS = {1: '#4393c3', 2: '#2ca25f', 3: '#fd8d3c', 4: '#de2d26'}
CLASS_LABELS = {
    'Amsterdam': {1: 'Pre-1945', 2: '1945-1984', 3: '1984-2004', 4: '2004-2024'},
    'Madrid':    {1: 'Pre-1960', 2: '1960-1984', 3: '1984-2004', 4: '2004-2024'},
}

print('Setup complete.')

### 3.2 Load Raw Parquet Datasets

In [ ]:
search_paths = ['.', './data', '../data', 'Building_Age_Classification/data', './Building_Age_Classification/data']
DATA_DIR = '.'
for p in search_paths:
    if os.path.exists(os.path.join(p, 'madrid_train.parquet')):
        DATA_DIR = p
        break

print(f"Using DATA_DIR: {DATA_DIR}")
df_madrid    = pd.read_parquet(os.path.join(DATA_DIR, 'madrid_train.parquet'))
df_amsterdam = pd.read_parquet(os.path.join(DATA_DIR, 'amsterdam_data.parquet'))

for name, df in [('Madrid', df_madrid), ('Amsterdam', df_amsterdam)]:
    print(f"{name}: {len(df):,} rows x {df.shape[1]} columns, {df.groupby(['px_key', 'py_key']).ngroups:,} unique pixels")

### 3.3 Age Class Assignment
Continuous `weighted_mean_year` is binned using historical boundaries.

In [ ]:
def assign_age_class(data, events=CLASS_EVENTS):
    out = data.copy()
    out['age_class'] = pd.NA
    for city, ev in events.items():
        mask = out['city'] == city
        if mask.any():
            bins = [-np.inf, ev, 1984, 2004, np.inf]
            out.loc[mask, 'age_class'] = pd.cut(
                out.loc[mask, 'weighted_mean_year'],
                bins=bins, labels=[1, 2, 3, 4]
            ).astype('Int64')
    return out

df_madrid    = assign_age_class(df_madrid)
df_amsterdam = assign_age_class(df_amsterdam)

for name, df in [('Madrid', df_madrid), ('Amsterdam', df_amsterdam)]:
    pix = df.drop_duplicates(['px_key', 'py_key'])
    print(f"{name} Labelled Pixels:")
    for cls in [1, 2, 3, 4]:
        n = (pix['age_class'] == cls).sum()
        print(f"  Class {cls} ({CLASS_LABELS[name][cls]}): {n:,} ({n/len(pix)*100:.1f}%)")

### 3.4 Multi-Observation Extraction & Atmospheric Cleaning
We extract the best valid observation per pixel-year, compute intra-annual variance across the 3 look slots, and remove haze outliers.

In [ ]:
def get_obs_stats(data, band):
    obs = []
    for slot in [1, 2, 3]:
        bc = f'{band}_{slot}'
        qc = f'qa_valid_{slot}'
        if bc in data.columns and qc in data.columns:
            val = data[bc].where(data[qc].fillna(False))
            obs.append(val)
    if not obs:
        return np.full(len(data), np.nan), np.full(len(data), np.nan)
    obs_stack = np.stack(obs, axis=1)
    best = np.nan
    for o in obs:
        best = np.where(np.isnan(best), o, best)
    var = np.nanvar(obs_stack, axis=1)
    return best, var

def make_flat_view(data):
    keep = [c for c in ['pixel_id', 'city', 'year', 'px_key', 'py_key', 'weighted_mean_year', 'age_class'] if c in data.columns]
    out = data[keep].copy()
    for band in BASE_BANDS:
        best, var = get_obs_stats(data, band)
        out[band] = best
        out[f'{band}_var'] = var
    out = out.dropna(subset=BASE_BANDS)
    n_before = len(out)
    out = out[out['Blue'] <= BLUE_MAX].copy()
    removed = n_before - len(out)
    if removed:
        city = data['city'].iloc[0] if 'city' in data.columns else ''
        print(f"  [{city}] Filtered {removed:,} haze outliers (Blue > {BLUE_MAX:,})")
    return out

print('Flattening observations...')
flat_madrid    = make_flat_view(df_madrid)
flat_amsterdam = make_flat_view(df_amsterdam)

# Period coverage indicators before gap fill
EARLY_CUTOFF = 2003
LATE_START   = 2004
PID          = ['px_key', 'py_key']

def period_coverage(flat):
    pixels = flat[PID].drop_duplicates()
    has_early = flat[flat['year'] <= EARLY_CUTOFF].drop_duplicates(PID).assign(has_early_data=1.0)
    has_late  = flat[flat['year'] >= LATE_START].drop_duplicates(PID).assign(has_late_data=1.0)
    return pixels.merge(has_early[PID + ['has_early_data']], on=PID, how='left') \
                 .merge(has_late[PID + ['has_late_data']], on=PID, how='left') \
                 .fillna(0.0)

indicators_madrid    = period_coverage(flat_madrid)
indicators_amsterdam = period_coverage(flat_amsterdam)
print('Flattening and coverage tracking completed.')

### 3.5 Temporal Gap-Filling (Linear Interpolation + Boundary Extrapolation)

In [ ]:
def fill_time_series_gaps(flat):
    year_min = flat['year'].min()
    year_max = flat['year'].max()
    all_years = list(range(year_min, year_max + 1))
    pixels = flat[PID + ['city', 'weighted_mean_year', 'age_class']].drop_duplicates(PID)
    grid = pixels.assign(_k=1).merge(pd.DataFrame({'year': all_years, '_k': 1}), on='_k').drop('_k', axis=1)
    core_bands = BASE_BANDS + VAR_BANDS
    merged = grid.merge(flat[PID + ['year'] + core_bands], on=PID + ['year'], how='left')
    merged = merged.sort_values(PID + ['year'])
    def interp(s):
        return s.interpolate(method='linear').bfill().ffill()
    for band in core_bands:
        merged[band] = merged.groupby(PID)[band].transform(interp)
    return merged

print('Filling gaps...')
flat_madrid    = fill_time_series_gaps(flat_madrid)
flat_amsterdam = fill_time_series_gaps(flat_amsterdam)
print('Gap filling complete.')

### 3.6 Spectral Indices & Spatial Neighborhood Morphology (KD-Tree)
We calculate 5 core multispectral indices (NDVI, NDBI, UI, MNDWI, BSI) and extract spatial context using pixel coordinates `(px_key, py_key)`.

In [ ]:
def compute_spectral_indices(df):
    df = df.copy()
    B, G, R = df['Blue'], df['Green'], df['Red']
    N, S1, S2 = df['NIR'], df['SWIR1'], df['SWIR2']
    eps = 1e-6
    df['NDVI']  = (N - R)  / (N + R  + eps)
    df['NDBI']  = (S1 - N) / (S1 + N + eps)
    df['UI']    = (S2 - N) / (S2 + N + eps)
    df['MNDWI'] = (G - S1) / (G + S1 + eps)
    df['BSI']   = ((S1 + R) - (N + B)) / ((S1 + R) + (N + B) + eps)
    return df

flat_madrid    = compute_spectral_indices(flat_madrid)
flat_amsterdam = compute_spectral_indices(flat_amsterdam)

def add_spatial_context(flat, radius=2):
    print(f"Computing spatial neighborhood context (radius={radius})...")
    pixel_means = flat.groupby(PID)[BASE_BANDS].mean().reset_index()
    coords = pixel_means[PID].to_numpy()
    arr = pixel_means[BASE_BANDS].to_numpy()
    tree = KDTree(coords)
    ind = tree.query_radius(coords, r=radius)
    spatial_arr = np.empty_like(arr)
    for i, neighbors in enumerate(ind):
        n_idx = neighbors[neighbors != i]
        spatial_arr[i] = arr[n_idx].mean(axis=0) if len(n_idx) > 0 else arr[i]
    spatial_df = pd.DataFrame(spatial_arr, columns=SPATIAL_BANDS)
    spatial_df[PID] = pixel_means[PID]
    return flat.merge(spatial_df, on=PID, how='left')

flat_madrid    = add_spatial_context(flat_madrid)
flat_amsterdam = add_spatial_context(flat_amsterdam)
print('Indices and spatial morphology engineered.')

### 3.7 Multi-Era Feature Aggregation (132 Dimensions)

In [ ]:
def build_pixel_features(flat):
    s = flat[flat['age_class'].notna()].copy().sort_values(PID + ['year'])
    target_cols = ALL_BANDS + INDEX_COLS
    grp = s.groupby(PID + ['age_class'])
    overall = pd.concat([
        grp[target_cols].mean().add_suffix('_mean'),
        grp[target_cols].std().add_suffix('_std')
    ], axis=1).reset_index()
    overall['age_class'] = overall['age_class'].astype(int)

    early_grp = s[s['year'] <= EARLY_CUTOFF].groupby(PID)
    early = pd.concat([
        early_grp[ALL_BANDS].mean().add_suffix('_early_mean'),
        early_grp[ALL_BANDS].std().add_suffix('_early_std')
    ], axis=1).reset_index()

    late_grp = s[s['year'] >= LATE_START].groupby(PID)
    late = pd.concat([
        late_grp[ALL_BANDS].mean().add_suffix('_late_mean'),
        late_grp[ALL_BANDS].std().add_suffix('_late_std')
    ], axis=1).reset_index()

    for b in BASE_BANDS:
        s[f'd_{b}'] = s.groupby(PID)[b].diff()
    dcols = [f'd_{b}' for b in BASE_BANDS]
    diff_grp = s.groupby(PID)[dcols]
    diff = pd.concat([
        diff_grp.mean().add_suffix('_mean'),
        diff_grp.std().add_suffix('_std')
    ], axis=1).reset_index()

    yr = s.groupby(PID)['weighted_mean_year'].first().reset_index()
    res = overall
    for extra in [early, late, diff, yr]:
        res = res.merge(extra, on=PID, how='left')
    return res

print('Building per-pixel features...')
features_madrid    = build_pixel_features(flat_madrid)
features_amsterdam = build_pixel_features(flat_amsterdam)

features_madrid    = features_madrid.merge(indicators_madrid,    on=PID, how='left')
features_amsterdam = features_amsterdam.merge(indicators_amsterdam, on=PID, how='left')

FEAT_OVERALL = [f'{b}_mean' for b in ALL_BANDS] + [f'{b}_std' for b in ALL_BANDS] + \
               [f'{i}_mean' for i in INDEX_COLS] + [f'{i}_std'  for i in INDEX_COLS]
FEAT_EARLY   = [f'{b}_early_mean' for b in ALL_BANDS] + [f'{b}_early_std' for b in ALL_BANDS]
FEAT_LATE    = [f'{b}_late_mean'  for b in ALL_BANDS] + [f'{b}_late_std'  for b in ALL_BANDS]
FEAT_YOY     = [f'd_{b}_mean' for b in BASE_BANDS] + [f'd_{b}_std' for b in BASE_BANDS]
FEAT_IND     = ['has_early_data', 'has_late_data']
FEATURE_COLS = FEAT_OVERALL + FEAT_EARLY + FEAT_LATE + FEAT_YOY + FEAT_IND

print(f"Total features engineered: {len(FEATURE_COLS)} (46 overall + 36 early + 36 late + 12 YOY + 2 indicators)")

### 3.8 Normalization & Pickle Export

In [ ]:
TARGET_COL = 'age_class'
X_madrid    = features_madrid[FEATURE_COLS].to_numpy(dtype=np.float64)
y_madrid    = features_madrid[TARGET_COL].to_numpy(dtype=int)
X_amsterdam = features_amsterdam[FEATURE_COLS].to_numpy(dtype=np.float64)
y_amsterdam = features_amsterdam[TARGET_COL].to_numpy(dtype=int)

scaler = StandardScaler()
X_madrid = scaler.fit_transform(X_madrid)
X_amsterdam = scaler.transform(X_amsterdam)

bundle = {
    'X_madrid': X_madrid,
    'y_madrid': y_madrid,
    'X_amsterdam': X_amsterdam,
    'y_amsterdam': y_amsterdam,
    'feature_names': FEATURE_COLS,
    'scaler': scaler,
    'pixel_ids_madrid': features_madrid[PID].to_numpy(),
    'pixel_ids_amsterdam': features_amsterdam[PID].to_numpy(),
    'feat_groups': {
        'overall': FEAT_OVERALL,
        'early': FEAT_EARLY,
        'late': FEAT_LATE,
        'yoy': FEAT_YOY,
        'indicators': FEAT_IND
    }
}

for out_p in ['preprocessed_data.pkl', 'data/preprocessed/preprocessed_data.pkl', '../data/preprocessed/preprocessed_data.pkl']:
    parent = os.path.dirname(out_p)
    if parent:
        os.makedirs(parent, exist_ok=True)
    try:
        with open(out_p, 'wb') as f:
            pickle.dump(bundle, f)
        print(f"Successfully saved bundle to {out_p}")
    except Exception as e:
        pass
print('Preprocessed data successfully saved and ready for UnitModel_4_Implementation.')